In [1]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

/content
Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 168 (delta 63), reused 96 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 885.91 KiB | 7.32 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/proto-tsrl
Mounted at /content/drive
/content/proto-tsrl


In [2]:
import functools

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.experiments.ppg.ppg_model import PPGModel
from src.utils.sampling_utils import *
from src.utils.training_utils import *

In [3]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data quality
INCLUDE_CLEAN_DATA = False
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False

# dataloaders
BATCH_SIZE = 256
VAL_RATIO = 0.05
N_WORKERS = 4

# contrastive sampling
COLLATE_FN_KWARGS = {
    'min_len': 100,
    'max_len': 300,
    'num_neg': 5,
    'min_overlap': 0.3,
    'min_var': 0.5,
    'max_var': 2.0
}

# schedule
N_EPOCHS_STAGE1 = 5
N_EPOCHS_STAGE2 = 5
N_EPOCHS_STAGE3 = 5
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed

# architecture
MASK_PROB = 0.2
MID_WEIGHT = 0.5
PROTO_NEG_MARGIN = 0.1
PROTO_DIVERSITY_THRESHOLD = 0.2
LAMBDA_PROTO = 1.0
TEMPERATURE = 1.0
LAMBDA_REPR = 1.0
GRADIENT_CLIP = None

REP_DIMS = [10, 50, 100]
MODELS = []
CKPT_PATHS = []
for dim in REP_DIMS:
    MODEL = PPGModel(representation_dimension = dim, mask_probability = MASK_PROB)
    if torch.cuda.device_count() > 1:
        MODEL = nn.DataParallel(MODEL)
    elif not isinstance(MODEL, nn.DataParallel):
        MODEL = nn.DataParallel(MODEL)

    SAVE_DIR = f"/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/dim{dim}"

    MODELS.append(MODEL)
    CKPT_PATHS.append(SAVE_DIR)

# optimizer
OPTIMIZERS = {
    "stage1": torch.optim.Adam(MODEL.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage2": torch.optim.Adam(MODEL.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage3": torch.optim.Adam(MODEL.parameters(), lr = 1e-4, weight_decay = 1e-5),
}

SCHEDULER = None

cuda:0


In [5]:
# DATA LOADING FUNCTIONS

# quality: high - [0,0,1] med - [0,1,0] low - [1,0,0]
# afib: pos - [0,1]
def load_data(
        clean : bool,
        seminoisy : bool,
        noisy : bool,
        dataset : str,
        return_labels : bool = True
    ):
    """
    Load PPG data from Drive

    Arguments
    ---------
        - clean, seminoisy, noisy: bools indicating whether to include data of each quality level
        - dataset: 'train' or 'test'
        - return_labels: whether to return labels (if False, only returns signals)

    Returns
    -------
        - X: (N, L) ndarray of PPG signals
        - y: (N,) ndarray of binary labels indicating presence of afib if return_labels = True
    """

    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data/'
    with np.load(file_path + f'{dataset}.npz') as data:
        ppg_signal = data['signal']
        ppg_qual = data['qa_label']
        rhythm = data['rhythm']

    qual_mask = np.zeros(ppg_qual.shape[0], dtype = bool)
    if clean:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 0, 1]), axis = 1))
    if seminoisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 1, 0]), axis = 1))
    if noisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([1, 0, 0]), axis = 1))

    X, y = ppg_signal[qual_mask], rhythm[qual_mask]

    afib_label = np.array([0, 1])
    y = np.all(y == afib_label, axis = 1)

    if return_labels:
        return X, y

    return X

def make_dataloaders(
        X : np.ndarray,
        y : np.ndarray = None,
        batch_size : int = 256,
        val_ratio : float = 0.05,
        seed : int = 42,
        num_workers : int = 4,
        collate_fn_kwargs : dict = None
    ) -> tuple[DataLoader, DataLoader]:
    """
    Create training and validation dataloaders from PPG signals and labels. If labels exist, they can be passed for use in stratified splitting.

    IMPORTANT: TimeSeriesDataset class takes signals as list of tensors of shape [T_i, F]

    Arguments
    ---------
        - X: (N, L) ndarray of PPG signals
        - y: (N,) ndarray of binary labels
        - batch_size: batch size for dataloaders
        - val_ratio: proportion of data to use for validation
        - seed: random seed for reproducibility
        - num_workers: number of worker processes for data loading
        - collate_fn_kwargs: dict of kwargs to pass to contrastive_collate function (if None, defaults will be used)

    Returns
    -------
        - dl_train: DataLoader for training data
        - dl_val: DataLoader for validation data
    """

    if y is not None:
        sss = StratifiedShuffleSplit(n_splits = 1, test_size = val_ratio, random_state = seed)
        train_idx, val_idx = next(sss.split(X, y))
    else:
        n = X.shape[0]
        indices = np.random.default_rng(seed).permutation(n)
        n_val = int(n * val_ratio)
        train_idx = indices[n_val:]
        val_idx = indices[:n_val]

    sig_train = [torch.from_numpy(x).float() for x in X[train_idx]]
    sig_val = [torch.from_numpy(x).float() for x in X[val_idx]]
    ds_train = TimeSeriesDataset(sig_train)
    ds_val = TimeSeriesDataset(sig_val)

    min_len = collate_fn_kwargs.get('min_len', X.shape[1]) if collate_fn_kwargs else X.shape[1]
    max_len = collate_fn_kwargs.get('max_len', X.shape[1]) if collate_fn_kwargs else X.shape[1]
    num_neg = collate_fn_kwargs.get('num_neg', 5) if collate_fn_kwargs else 4
    min_overlap = collate_fn_kwargs.get('min_overlap', 0.3) if collate_fn_kwargs else 0.3
    min_var = collate_fn_kwargs.get('min_var', 0.5) if collate_fn_kwargs else 0.5
    max_var = collate_fn_kwargs.get('max_var', 2) if collate_fn_kwargs else 2

    collate_fn = functools.partial(
        contrastive_collate,
        min_len = min_len,
        max_len = max_len,
        num_neg = num_neg,
        min_overlap = min_overlap,
        min_var = min_var,
        max_var = max_var
    )

    dl_train = DataLoader(
        ds_train,
        batch_size = batch_size,
        shuffle = True,
        collate_fn = collate_fn,
        num_workers = num_workers,
        pin_memory = True,
        drop_last = True
    )

    dl_val = DataLoader(
        ds_val,
        batch_size = batch_size,
        shuffle = False,
        collate_fn = collate_fn,
        num_workers = num_workers,
        pin_memory = True,
        drop_last = True
    )

    return dl_train, dl_val

In [6]:
# LOAD DATA

X_train, y_train = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)

indices = np.random.default_rng(SEED).permutation(X_train.shape[0])
X_train, y_train = X_train[indices][:100000], y_train[indices][:100000]

X_test, y_test = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'test',
    return_labels = True
)

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')
print(f'X_test shape: {X_test.shape}')
print(f'Test set positive samples: {np.sum(y_test)}')

dl_train, dl_val = make_dataloaders(
    X_train,
    y_train,
    batch_size = BATCH_SIZE,
    val_ratio = VAL_RATIO,
    seed = SEED,
    num_workers = N_WORKERS,
    collate_fn_kwargs = COLLATE_FN_KWARGS
)

X_train shape: (100000, 800, 1)
Train set positive samples: 50193
X_test shape: (2032, 800, 1)
Test set positive samples: 314


In [7]:
### TRAINING

for i, dim in enumerate(REP_DIMS):
    print(f'TRAINING MODEL WITH REPRESENTATION DIMENSION {dim}')
    history = run_training(
        model = MODELS[i],
        train_loader = dl_train,
        val_loader = dl_val,
        device = DEVICE,
        optimizer_dict = OPTIMIZERS,
        epochs_stage1 = N_EPOCHS_STAGE1,
        epochs_stage2 = N_EPOCHS_STAGE2,
        epochs_stage3 = N_EPOCHS_STAGE3,
        scheduler_dict = SCHEDULER,
        mid_weight = MID_WEIGHT,
        proto_neg_margin = PROTO_NEG_MARGIN,
        proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
        lambda_proto = LAMBDA_PROTO,
        temperature = TEMPERATURE,
        lambda_repr = LAMBDA_REPR,
        grad_clip_norm = GRADIENT_CLIP,
        checkpoint_path = CKPT_PATHS[i],
        checkpoint_epochs = CHECKPOINT_EPOCHS,
        collector_fn = None,
        use_amp = True
    )

TRAINING MODEL WITH REPRESENTATION DIMENSION 100
Training for 15 epochs.

=== stage1 (5 epochs) ===
[stage1] epoch 1/5 | global 1/15
  train loss: 3.951092 | val loss: 3.491832
Saved checkpoint at epoch 1
[stage1] epoch 2/5 | global 2/15
  train loss: 3.951401 | val loss: 3.626245
Saved checkpoint at epoch 2
[stage1] epoch 3/5 | global 3/15
  train loss: 3.954231 | val loss: 3.549410
Saved checkpoint at epoch 3
[stage1] epoch 4/5 | global 4/15
  train loss: 3.953656 | val loss: 3.583231
Saved checkpoint at epoch 4
[stage1] epoch 5/5 | global 5/15
  train loss: 3.951655 | val loss: 3.562381
Saved checkpoint at epoch 5

=== stage2 (5 epochs) ===
[stage2] epoch 1/5 | global 6/15
  train loss: 45.599069 | val loss: 45.489054
Saved checkpoint at epoch 6
[stage2] epoch 2/5 | global 7/15
  train loss: 45.600729 | val loss: 45.486165
Saved checkpoint at epoch 7
[stage2] epoch 3/5 | global 8/15
  train loss: 45.602960 | val loss: 45.477570
Saved checkpoint at epoch 8
[stage2] epoch 4/5 | global

In [8]:
runtime.unassign()